# Machine Learning Exercise

Now it's time to practice what you have seen in the previous notebooks. Your task for today is to download the data from the database and train a model in order to predict if a patient has a heart disease or not. 

![](https://www.nicepng.com/png/detail/397-3975460_disease-high-quality-png-heart-disease-cartoon-png.png)

## Step 1 - Define the Goal and Research Question

1. Import the data from the database. The schema is called `heart`. You can use DBeaver to get an overview over the different tables and think about a good way to join them. 
2. Conduct a brief EDA to become familiar with the data. 
3. Preprocess the data as far as you need it and...
4. ...train a logistic regression model.

### Success Metric

We decide how to measure success before modelling. This is a medical screening task, so a **false negative** (missing a real heart attack) is far costlier than a **false positive** (a false alarm). Plain accuracy treats both errors the same, so we select and tune on the **F2 score** of the heart-attack class: an F-beta score with beta = 2, which weights recall (catching true cases) about twice as heavily as precision. For each model we report accuracy and F2 and, most usefully, the confusion matrix, which shows the false negatives and false positives directly.

### What to Use and Keep in Mind
 
* **Scale your data:** Which scaler works best in your case?
* **Tune your model:** Tune the hyperparameter of your model. You can start with a larger parameter grid and a `RandomizedSearchCV` and continue with a narrower parameter grid for your `GridSearchCV`.
* **Choose the right evaluation metric!**


### Data Overview

| column | additional information |
|--------|------------------------|
| age | age of patient |
| sex | gender of patient |
| chest_pain_type  | 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic | 
| resting_blood_pressure |  | 
| fasting_blood_sugar | > 120 mg/dl, 1 = true, 0 = false | 
| thal | 0 = normal, 1 = fixed defect, 2 = reversable defect
| serum_cholestoral | in mg/dl | 
| resting_electrocardiographic_results | 0 = normal, 1 = having ST-T wave abnormality, 2 = showing probable or definite left ventricular hypertrophy by Estes' criteria | 
| maximum_heartrate_achieved | | 
| exercise_induced_angina | 1 = yes, 0 = no | 
| oldpeak | ST depression induced by exercise relative to rest | 
| slope_of_the_peak_exercise_st_segment | 1= upsloping, 2 = flat, 3 = downsloping | 
| number_of_major_vessels_colored_by_flourosopy | |
| real_data | tag to distinguish between real and made up data | 
| heart_attack | 0 = little risk of heart attack, 1 = high risk of heart attack | 

## Step 2 - Get the Data

We import the libraries we need, then pull the heart data from the database.

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

import numpy as np
import pandas as pd


# Modelling
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import fbeta_score, make_scorer, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from scipy.stats import loguniform, uniform

# Plot
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
plt.rcParams["figure.facecolor"] = "w"
pd.plotting.register_matplotlib_converters()
pd.set_option("display.float_format", lambda x: "%.3f" % x)
pd.options.display.float_format = "{:,.2f}".format


RSEED = 42

### Load from the Database

The data for this exercise is stored in our postgres database in the schema `heart`. The different features are split thematically into five different tables. Your first task will be to have a look at the tables (e.g. in DBeaver) and figure out a way to join the information you need. As soon as you're happy with your query, you can use the following code cells to import the data into this notebook. 

In previous notebooks you've seen two different approaches to import data from a database into a notebook. The following code will use `sqlalchemy`in combination with pandas `pd.read_sql()` function. For the code to work, you need to copy the `.env` file from the previous repositories into this repository and change the query_string to your own query.

In [ ]:
# Read database string from .env file (no need to change anything)
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

db = create_engine(DB_STRING)

In [ ]:
# Define query to download data (add your query here)
query_string = """ 
set SCHEMA 'heart';
SELECT p.id as patient_id,p.age,p.sex, 
cp.chest_pain_type,
bm.serum_cholestoral, bm.fasting_blood_sugar,bm.thal, 
pva.resting_blood_pressure,pva.resting_electrocardiographic_results,pva.maximum_heartrate_achieved,pva.exercise_induced_angina,pva.oldpeak,pva.slope_of_the_peak_exercise_st_segment,
pva.number_of_major_vessels_colored_by_flourosopy,pva.real_data,
hah.heart_attack
FROM patient p
LEFT JOIN chest_pain cp
ON p.id = cp.patient_id
LEFT JOIN blood_metrics bm
ON p.id = bm.patient_id
LEFT JOIN pressure_vessels_angina pva
ON p.id = pva.patient_id
LEFT JOIN heart_attack_history hah
ON p.id = hah.patient_id
WHERE pva.real_data <> 'Evgeny likes white wine for lunch and red wine for dinner' or pva.real_data is NULL
ORDER BY p.id ASC;
"""

# Import with pandas
df_sqlalchemy = pd.read_sql(query_string, db)
df_sqlalchemy.head()

In [ ]:
if not os.path.exists("./data"):
    os.mkdir("data")

In [ ]:
# Save dataframe as .csv file
df_sqlalchemy.to_csv("data/heart_data.csv", index=False)

In [ ]:
df = pd.read_csv("data/heart_data.csv")

In [ ]:
df.head()

In [ ]:
# Print the shape of the data
print("Heart dataset")
print("==================")
print("# observations: {}".format(df.shape[0]))
print("# features:     {}".format(df.shape[1] - 1))

### Structural Cleanup (before the split)

We first do structural cleaning that learns nothing from the data, so it is safe to apply before splitting: renaming the columns, and checking for and removing exact duplicate rows. Removing duplicates here (before the split) ensures the same observation cannot end up in both the train and test sets.

In [ ]:
# Renaming the column names

df.rename(
    columns={
        "fasting_blood_sugar": "fasting_bs",
        "resting_blood_pressure": "resting__bp",
        "resting_electrocardiographic_results": "electrocardiography",
        "maximum_heartrate_achieved": "max_heartrate",
        "exercise_induced_angina": "exercise_ind_angina",
        "slope_of_the_peak_exercise_st_segment": "slope_of_peak_exercise",
        "number_of_major_vessels_colored_by_flourosopy": "major_vessels_colored",
    },
    inplace=True,
)

In [ ]:
# Checking for duplicates


def check_duplicates(data):
    has_dup = data.duplicated()
    n_dup = has_dup.sum()

    if n_dup > 0:
        print("Data has", n_dup, "duplicates")
    else:
        print("Data has no duplicates")


check_duplicates(df)

In [ ]:
# Drop exact duplicate rows before splitting, so the same observation
# cannot end up in both the train and test sets.
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows; {len(df)} rows remain.")

## Step 3 - Train-Test Split

We split now, before any exploration or fitted preprocessing. This keeps the test set genuinely unseen, so it cannot leak into the choices we make while exploring or while fitting transformers. Every step below uses the training data only.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, shuffle=True)

print("train_df:", train_df.shape)
print("test_df: ", test_df.shape)

## Step 4 - Data Exploration

All exploration below runs on `train_df`. If you split your features and target separately instead, you can rebuild a single frame for EDA with `pd.concat([X_train, y_train], axis=1)`.

### Understand the Basics

In [ ]:
# Type and non-null counts
train_df.info()

In [ ]:
# Descriptive statistics
train_df.describe().T

In [ ]:
# Unique values per column
train_df.nunique()

### Check for Missing Values

In [ ]:
# Share of missing values per column
train_df.isnull().mean()

In [ ]:
# Class balance of the target
train_df["heart_attack"].value_counts()

### Visualize the Data

In [ ]:
# Plot distribution of features
features = train_df.columns.tolist()
features.remove("heart_attack")

fig, ax = plt.subplots(8, 3, figsize=(34, 30))
count = 0
for item in features:
    sns.histplot(
        train_df[item], kde=True, ax=ax[int(count / 3)][count % 3], color="#33658A"
    ).set(title=item, xlabel="")
    count += 1
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

The target is well balanced, not deviating by much from a 50% / 50% split.

### Identify Outliers

In [ ]:
# Checking for Outliers without heart attack column
df_outliers = train_df[
    [
        "age",
        "sex",
        "chest_pain_type",
        "serum_cholestoral",
        "fasting_bs",
        "thal",
        "resting__bp",
        "electrocardiography",
        "max_heartrate",
        "exercise_ind_angina",
        "oldpeak",
        "slope_of_peak_exercise",
        "major_vessels_colored",
        "real_data",
    ]
].copy()

df_outliers.plot(kind="box", subplots=True, layout=(8, 3), figsize=(34, 30))
plt.show()  #

### Decide How to Handle the Missing Values

We recode `real_data` into a simple flag and compare the other variables for rows where it is present versus missing.

In [ ]:
train_df.real_data = np.where(train_df.real_data == "real data", "real data", "null")

In [ ]:
sns.pairplot(train_df, hue="real_data", height=2);

The distributions look very similar: they overlap and peak at the same values. Visual inspection is enough here to treat the `real_data` nulls as legitimate.

Now let's inspect how the variables differ depending on the target variable, `heart_attack`.

In [ ]:
sns.pairplot(train_df, hue="heart_attack", height=2);

There are some clear differences here: variables with large differences include `chest_pain_type`, `thal`, and `electrocardiography`.

### Correlation Analysis

In [ ]:
# Correlation heatmap

fig, ax = plt.subplots(figsize=(12, 8))

# Create a new DataFrame that only includes the numerical variables
df_numeric = train_df.select_dtypes(include=["float64", "int64"])

# Compute correlations
correlations = df_numeric.corr()

# Generate a mask for the upper triangle
mask = np.zeros_like(correlations)
mask[np.triu_indices_from(mask)] = True

# Generate a custom diverging colormap
cmap = sns.diverging_palette(140, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio

sns.heatmap(
    correlations,
    mask=mask,
    cmap=cmap,
    vmax=1,
    annot=True,
    linewidths=1,
    cbar_kws={"shrink": 0.7},
    ax=ax,
);

The heatmap shows how strongly each numeric feature moves with `heart_attack`. Setting aside `patient_id` (whose unusually strong correlation we investigate later), the strongest relationships are with `max_heartrate` (positive), `major_vessels_colored`, `chest_pain_type`, and `oldpeak`. In particular, a higher maximum heart rate achieved is associated with having had a heart attack.

## Step 5 - Feature Engineering

Now we prepare the features. First a little row-level cleaning (drop rows with missing feature values, and apply the `real_data` recode to the test set), then we define the feature groups and build a `ColumnTransformer` that scales the numeric columns and one-hot encodes the categorical ones.

In [ ]:
# real_data was recoded on the training set above; apply the same flag to test
test_df.real_data = np.where(test_df.real_data == "real data", "real data", "null")


# Drop rows with missing values in any column except real_data (its nulls are legitimate)
def drop_feature_nulls(d):
    mask = d.drop(columns="real_data").isnull().any(axis=1)
    return d[~mask].copy()


train_df = drop_feature_nulls(train_df)
test_df = drop_feature_nulls(test_df)

print("train_df:", train_df.shape, " test_df:", test_df.shape)

### Define Features and Target

In [ ]:
# Group the feature columns by how they should be engineered
num_cols = [
    "age",
    "sex",
    "serum_cholestoral",
    "fasting_bs",
    "resting__bp",
    "max_heartrate",
    "exercise_ind_angina",
    "oldpeak",
    "major_vessels_colored",
]
cat_cols = [
    "chest_pain_type",
    "thal",
    "electrocardiography",
    "slope_of_peak_exercise",
    "real_data",
]
feature_cols = num_cols + cat_cols

X_train, y_train = train_df[feature_cols], train_df["heart_attack"]
X_test, y_test = test_df[feature_cols], test_df["heart_attack"]

# Versions that also keep patient_id, used to show the leakage it causes
X_train_id = train_df[feature_cols + ["patient_id"]]
X_test_id = test_df[feature_cols + ["patient_id"]]

print("X_train:", X_train.shape, " X_test:", X_test.shape)

### Build the Feature-Engineering Pipeline

Following the first notebook, all feature engineering goes into a single `ColumnTransformer` that scales the numeric columns and one-hot encodes the categorical ones. We wrap it with the model in a `Pipeline`, so each transformer is fit on the training folds only, even inside cross-validation. The helper below builds that pipeline for a given scaler.

In [ ]:
def build_pipeline(scaler="passthrough", numeric=num_cols):
    # One ColumnTransformer does all feature engineering: scale the numeric
    # columns and one-hot encode the categorical ones. handle_unknown='ignore'
    # encodes categories unseen during fit as all zeros instead of raising.
    feature_engineering = ColumnTransformer(
        [
            ("num", scaler, numeric),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
        ]
    )
    return Pipeline(
        [
            ("features", feature_engineering),
            ("model", LogisticRegression(max_iter=5000)),
        ]
    )

**A note on the warning you will see below.** When the pipeline transforms data that contains a category not seen while fitting, `OneHotEncoder(handle_unknown="ignore")` encodes it as all zeros and prints `Found unknown categories in columns [...] during transform`. With this small dataset, some rare category values land only in the test set or in a particular cross-validation fold, so this happens often. Encoding them as all zeros is exactly the safe behavior we asked for, so the message is informational, not an error.

## Steps 6 to 8 - Model Training and Hyperparameter Tuning

Feature engineering, model training, hyperparameter tuning, and validation form an iterative loop. We start from a simple baseline, then improve it with scaling and tuned hyperparameters.

In [ ]:
# Helper to evaluate each model: print accuracy and our chosen F2 score, then show the
# confusion matrix so we can read the false negatives and false positives directly.
def evaluate(name, y_true, y_pred):
    print(f"{name}: accuracy {accuracy_score(y_true, y_pred):.2f} | F2 {fbeta_score(y_true, y_pred, beta=2):.2f}")
    plt.figure(figsize=(4, 3))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="YlGnBu")
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

### Baseline Model

Before building anything complex, we establish a simple heuristic baseline as a benchmark the real model should beat. The correlation analysis above showed that, among the legitimate features, `max_heartrate` has the strongest positive correlation with the target: patients with a higher maximum heart rate achieved are more likely to have had a heart attack. So our baseline predicts 1 when `max_heartrate` is above a threshold, which we read from the training data (the midpoint between the two class averages).

In [ ]:
# Derive the threshold from the training data: the midpoint between the average
# max_heartrate of each class. Above it, a patient looks more like the heart-attack group.
class_means = train_df.groupby("heart_attack")["max_heartrate"].mean()
threshold = class_means.mean()
print("Average max_heartrate per class:")
print(class_means.round(1).to_string())
print("Threshold (midpoint):", round(threshold, 1))

rule_pred_test = (X_test["max_heartrate"] > threshold).astype(int)
evaluate("Baseline rule", y_test, rule_pred_test)

### First Logistic Regression (and a Leakage Trap)

Now a real model. We fit a logistic regression through the pipeline, first without `patient_id` and then with it, to show how an identifier can leak the target.

In [ ]:
# Logistic regression: numeric columns passed through unscaled, categoricals one-hot encoded
log_reg = build_pipeline()
log_reg.fit(X_train, y_train)
evaluate("Logistic regression", y_test, log_reg.predict(X_test))

In [ ]:
# Same pipeline, but patient_id is added to the numeric features
log_reg_id = build_pipeline(numeric=num_cols + ['patient_id'])
log_reg_id.fit(X_train_id, y_train)
evaluate("Logistic regression + patient_id", y_test, log_reg_id.predict(X_test_id))

Including `patient_id` pushes the scores close to 1 and leaves almost nothing off the diagonal of the confusion matrix, which makes no real sense. Let's look more closely at what is happening.

In [ ]:
sns.kdeplot(data=train_df, x="patient_id", hue="heart_attack", fill=True);

The two classes barely overlap on the id axis, which is suspicious. Let's check it more closely.

In [ ]:
_ = sns.scatterplot(data=train_df, x="patient_id", y="heart_attack")

In [ ]:
# Smallest id with heart_attack == 1
train_df.query("heart_attack == 1").patient_id.min()

In [ ]:
# Smallest id with heart_attack == 0
train_df.query("heart_attack == 0").patient_id.min()

In [ ]:
# Smallest id with heart_attack == 1 above 165
train_df.query("heart_attack == 1 and patient_id > 165").patient_id.min()

Patients with ids up to about 165 all had heart attacks, ids from 165 to 303 had none, and above 303 the id no longer signals the outcome. Whoever entered the data created this pattern, but there is no reason to expect future ids to follow it, so `patient_id` must not be used as a feature.

### Training with Feature Scaling

Now we plug a scaler into the same pipeline and compare standardization with normalization.

#### Standardization

In [ ]:
std_pipe = build_pipeline(StandardScaler())
std_pipe.fit(X_train, y_train)
evaluate("Standardized", y_test, std_pipe.predict(X_test))

#### Normalization (MinMaxScaler)

In [ ]:
minmax_pipe = build_pipeline(MinMaxScaler())
minmax_pipe.fit(X_train, y_train)
evaluate("Normalized (MinMaxScaler)", y_test, minmax_pipe.predict(X_test))

### Hyperparameter Tuning with RandomizedSearchCV

We tune the whole pipeline, so the scaler is refit inside every cross-validation fold. Parameter names are prefixed with the pipeline step they belong to (`model__`).

In [ ]:
# We tune on the F2 score (recall-weighted), the success metric defined in Step 1
f2_scorer = make_scorer(fbeta_score, beta=2)

search_pipe = build_pipeline(StandardScaler())

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

# saga supports the full L1/L2 range through l1_ratio (0 = L2, 1 = L1)
param_distributions = {
    "model__solver": ["saga"],
    "model__C": loguniform(1e-5, 100),
    "model__l1_ratio": uniform(0, 1),
}

random_search = RandomizedSearchCV(
    search_pipe, param_distributions, n_iter=50, scoring=f2_scorer,
    n_jobs=-1, cv=cv, random_state=1
)
random_search.fit(X_train, y_train)
pred_test = random_search.predict(X_test)
print("Best parameters:", random_search.best_params_)
print(f"Test: accuracy {accuracy_score(y_test, pred_test):.2f} | F2 {fbeta_score(y_test, pred_test, beta=2):.2f}")

### Hyperparameter Tuning with GridSearchCV

`np.logspace` generates values evenly spaced on a log scale to search over `C`. We also vary `l1_ratio` between 0 (an L2 penalty) and 1 (an L1 penalty); the `saga` solver supports the full range.

In [ ]:
# l1_ratio = 0 is an L2 penalty, l1_ratio = 1 is an L1 penalty; saga supports both
param_grid = {
    "model__solver": ["saga"],
    "model__C": np.logspace(0, 4, num=10),
    "model__l1_ratio": [0, 1],
}

grid_search = GridSearchCV(
    build_pipeline(StandardScaler()), param_grid, scoring=f2_scorer,
    cv=5, n_jobs=-1
)
grid_search.fit(X_train, y_train)
pred_test = grid_search.predict(X_test)
print("Best parameters:", grid_search.best_params_)
print(f"Test: accuracy {accuracy_score(y_test, pred_test):.2f} | F2 {fbeta_score(y_test, pred_test, beta=2):.2f}")

## Step 9 - Calculate Test Score

We now evaluate the tuned model on the held-out test set using the metric chosen in Step 1 (F2), alongside the full classification report and confusion matrix so we can see precision, recall, and the actual error counts. In a stricter workflow you would refit the chosen model on the combined training and validation data before this final evaluation, so the test set is touched only once.

In [ ]:
# Final evaluation of the tuned model on the held-out test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
evaluate("Final tuned model", y_test, y_pred)
print(classification_report(y_test, y_pred))

## Step 10 - Deploy and Monitor

Deployment and monitoring are beyond the scope of this exercise. In practice you would put the chosen pipeline into production and watch its performance over time, retraining if it degrades.